!pip3 install drain3 pandas regex


In [1]:
!pip3 install drain3 pandas regex


  Installing build dependencies ...   Installing build dependencies ... -done
done
  Getting requirements to build wheel ...   Getting requirements to build wheel ... -done
  Preparing metadata (pyproject.toml) ... one
  Preparing metadata (pyproject.toml) ... -done
one
  Created wheel for drain3: filename=drain3-0.9.11-py3-none-any.whl size=24086 sha256=25bf8aca9120dd869456b787515ee178abaed1593309414496a1ae0eed321f53
  Stored in directory: /Users/michalklos/Library/Caches/pip/wheels/96/3f/bb/c2df80298168b46a45654266ac0c139220540689a17463e3cf
Successfully built drain3
done
  Created wheel for drain3: filename=drain3-0.9.11-py3-none-any.whl size=24086 sha256=25bf8aca9120dd869456b787515ee178abaed1593309414496a1ae0eed321f53
  Stored in directory: /Users/michalklos/Library/Caches/pip/wheels/96/3f/bb/c2df80298168b46a45654266ac0c139220540689a17463e3cf
Successfully built drain3

[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[n

In [4]:
# Install Drain3

import os
import pandas as pd
import regex as re
import urllib.request
import ssl
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# Download HDFS_2k.log sample from Loghub
data_url = "https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log"
log_file = "HDFS_2k.log"

if not os.path.exists(log_file):
    print(f"Downloading {log_file}...")
    # Create unverified SSL context to bypass certificate errors
    ssl_context = ssl._create_unverified_context()
    with urllib.request.urlopen(data_url, context=ssl_context) as response, open(log_file, 'wb') as out_file:
        out_file.write(response.read())
    print("Download complete.")
else:
    print(f"{log_file} already exists.")

# Display first few lines
with open(log_file, 'r') as f:
    head = [next(f) for _ in range(5)]
print("Sample logs:")
print("".join(head))

Download complete.
Sample logs:
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
081109 204015 308 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_8229193803249955061 terminating
081109 204106 329 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-6670958622368987959 terminating

Download complete.
Sample logs:
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock

In [6]:
# Configure Drain3
config = TemplateMinerConfig()
# config.load_default_config() # Removed: Not needed/available in this version
config.profiling_enabled = False
template_miner = TemplateMiner(config=config)

# Regex to extract BlockId (HDFS specific)
block_id_pattern = re.compile(r'(blk_[-0-9]+)')

parsed_data = []

print("Parsing logs...")
with open(log_file, 'r') as f:
    for line_count, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
            
        # 1. Extract BlockId
        match = block_id_pattern.search(line)
        block_id = match.group(1) if match else "Unknown"
        
        # 2. Parse with Drain3
        # We process the message part. For HDFS, the content usually starts after some timestamp/level info.
        # However, Drain3 can handle full lines, but it's better to strip headers if possible.
        # For simplicity, we'll pass the full line or a simple split.
        # HDFS format: <Date> <Time> <Pid> <Level> <Component>: <Content>
        # Example: 081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
        
        # Simple heuristic: take everything after the first colon if present, else full line
        content = line.split(':', 1)[1].strip() if ':' in line else line
        
        result = template_miner.add_log_message(content)
        
        parsed_data.append({
            'LineId': line_count + 1,
            'BlockId': block_id,
            'RawContent': content,
            'EventId': result['cluster_id'],
            'EventTemplate': result['template_mined']
        })

print(f"Finished parsing {len(parsed_data)} lines.")
print(f"Found {len(template_miner.drain.clusters)} unique templates.")

# Convert to DataFrame
df_parsed = pd.DataFrame(parsed_data)
df_parsed.head()

Parsing logs...
Finished parsing 2000 lines.
Found 17 unique templates.


,LineId,BlockId,RawContent,EventId,EventTemplate
0,1,blk_38865049064139660,PacketResponder 1 for block blk_38865049064139...,1,PacketResponder 1 for block blk_38865049064139...
1,2,blk_-6952295868487656571,PacketResponder 0 for block blk_-6952295868487...,1,PacketResponder <*> for block <*> terminating
2,3,blk_7128370237687728475,BLOCK* NameSystem.addStoredBlock: blockMap upd...,2,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,blk_8229193803249955061,PacketResponder 2 for block blk_82291938032499...,1,PacketResponder <*> for block <*> terminating
4,5,blk_-6670958622368987959,PacketResponder 2 for block blk_-6670958622368...,1,PacketResponder <*> for block <*> terminating


In [7]:
# Display extracted templates
print("Extracted Templates:")
templates_df = df_parsed[['EventId', 'EventTemplate']].drop_duplicates().sort_values('EventId')
for _, row in templates_df.iterrows():
    print(f"ID {row['EventId']}: {row['EventTemplate']}")

# Create Log Sequences
# Group by BlockId and collect EventIds
sequence_df = df_parsed.groupby('BlockId')['EventId'].apply(list).reset_index()
sequence_df.rename(columns={'EventId': 'EventSequence'}, inplace=True)

# Calculate sequence length
sequence_df['Length'] = sequence_df['EventSequence'].apply(len)

print("\nGenerated Sequences:")
print(sequence_df.head())

# Example of a full sequence
sample_block = sequence_df.iloc[0]
print(f"\nBlock: {sample_block['BlockId']}")
print(f"Sequence: {sample_block['EventSequence']}")
print("Reconstructed Sequence (Templates):")
for eid in sample_block['EventSequence']:
    template = templates_df[templates_df['EventId'] == eid]['EventTemplate'].values[0]
    print(f"  -> {template}")

Extracted Templates:
ID 1: PacketResponder 1 for block blk_38865049064139660 terminating
ID 1: PacketResponder <*> for block <*> terminating
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size 67108864
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
ID 3: Received block blk_3587508140051953248 of size 67108864 from /10.251.42.84
ID 3: Received block <*> of size 67108864 from <*>
ID 3: Received block <*> of size <*> from <*>
ID 4: Receiving block blk_5792489080791696128 src: /10.251.30.6:33145 dest: /10.251.30.6:50010
ID 4: Receiving block <*> src: <*> dest: <*>
ID 5: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000590_0/part-00590. blk_-1727475099218615100
ID 5: BLOCK* NameSystem.allocateBlock: <*> <*>
ID 6: Verification succeeded for blk_-49809165198